# Extract Colab Environment → requirements.txt

Run this notebook **in Google Colab** after your FED notebooks (e.g. SFT_LoRA_Adapters) work. It installs the same packages, captures exact versions, and saves `colab_requirements.txt` for use in JupyterHub.

**Runtime:** Use T4 GPU if you need GPU packages.

## 1. Install packages (same as FED/DS1000 SFT notebook)

In [ ]:
!pip install --no-cache-dir "transformers>=4.36" "peft>=0.7" "bitsandbytes>=0.41" "trl>=0.7,<0.20" "datasets" "accelerate" "pandas"
# Fix: torchao 0.15+ uses torch.int1/int2/... (PyTorch 2.5+ only)
!pip install -q --no-cache-dir "torchao<0.15"
# Ensure PyTorch >= 2.4
!pip install -q --no-cache-dir "torch>=2.4" "torchvision" "torchaudio" --upgrade

## 2. Capture environment and save requirements

In [ ]:
import subprocess
import sys

out = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode("utf-8")

# Full freeze for exact reproducibility; skip Colab-only packages (pip -r will ignore if missing)
lines = [ln for ln in out.strip().split("\n") if ln.strip()]
SKIP = ("google-colab", "colabtools")
filtered = [ln for ln in lines if not any(ln.lower().startswith(p) for p in SKIP)]

req_content = "\n".join(filtered)
with open("/content/colab_requirements.txt", "w") as f:
    f.write(req_content)

print("Saved", len(filtered), "packages to /content/colab_requirements.txt")
print("\n--- Environment info ---")
print("Python:", sys.version.split(" ")[0])
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch:", e)
print("\nFirst 20 lines of requirements:")
for line in filtered[:20]:
    print(" ", line)

## 3. Download requirements file

In [ ]:
try:
    from google.colab import files
    files.download("/content/colab_requirements.txt")
    print("Download started: colab_requirements.txt")
except ImportError:
    print("Not in Colab. File is at /content/colab_requirements.txt")